# 02 - Construcao da base analitica anual (carreta x ano)

**Fonte unica de dados:** `data/raw/fato_wo_ml_2020-01-01_to_2025-12-31.csv`.

Este notebook parte **exclusivamente** da base consolidada de ordens de servico (OS)
enriquecida (`fato_wo_ml`). A extracao SQL, os *joins* do modelo estrela e o
*feature engineering* que originaram esse arquivo pertencem a uma **etapa anterior
de preparacao dos dados**; o presente estudo inicia sua analise a partir do arquivo
consolidado e nao reconstroi a base nem executa novos *joins*.

A partir das OS (grao: 1 linha = 1 ordem de servico), constroi-se a base analitica
no **grao carreta x ano**, cuja variavel resposta e o **custo anual de manutencao por
carreta**. A correcao monetaria (CPI Canada) e aplicada no notebook `04`; aqui os
custos ainda estao em valores **nominais** (CAD).


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
TABLES = REPORTS / "tables"
FIGURES = REPORTS / "figures"
for d in (DATA_PROCESSED, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

CSV = DATA_RAW / "fato_wo_ml_2020-01-01_to_2025-12-31.csv"
ANO_MIN, ANO_MAX = 2020, 2025
pd.set_option("display.width", 120)
print("Fonte unica:", CSV.name)


Fonte unica: fato_wo_ml_2020-01-01_to_2025-12-31.csv


## 1. Carga da base consolidada e curadoria minima

Regras de curadoria (registradas em `reports/tables/02_curadoria_limpeza.csv`):
- restringe a janela do estudo a **2020-2025** (descarta poucas OS carimbadas fora dela);
- exclui OS com **custo interno negativo** (estornos), coerente com a decisao historica do projeto;
- normaliza nomes de colunas e converte tipos (datas e valores monetarios).

In [2]:
raw = pd.read_csv(CSV, low_memory=False)
raw.columns = [c.strip().lower() for c in raw.columns]

raw["data_os"] = pd.to_datetime(raw["data_os"], errors="coerce")
raw["data_entrada_servico"] = pd.to_datetime(raw["data_entrada_servico"], errors="coerce")
raw["ano"] = raw["data_os"].dt.year
raw["mes"] = raw["data_os"].values.astype("datetime64[M]")
for c in ["total_custo_interno", "km_acumulado_data_os", "ano_modelo", "eixos", "comprimento",
          "tempo_contrato_meses_ate_reparo"]:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

# normaliza categoricas: espacos em branco / vazios -> NaN
CAT_ATTRS = ["descricao_carreta", "cod_montadora", "flag_refrigerado", "tailgate_flag",
             "unit_subtype", "tire_size", "suspension_type", "new_used_indicator"]
# contrato (2026-08-16): os 4 campos passaram a integrar a fonte unica.
# franquia_km_mensal_contrato NAO e usada (D3: 99,8% dos preenchidos = 0, variancia nula).
for c in CAT_ATTRS + ["provincia_estado", "cod_local_os", "vmrs", "tipo_manutencao", "cod_cliente"]:
    raw[c] = raw[c].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})

n0 = len(raw)
fora_janela = ~raw["ano"].between(ANO_MIN, ANO_MAX)
estorno = raw["total_custo_interno"] < 0
wo = raw[~fora_janela & ~estorno].copy()

curadoria = pd.DataFrame([
    {"passo": "linhas_originais", "linhas": n0},
    {"passo": "removidas_fora_janela_2020_2025", "linhas": int(fora_janela.sum())},
    {"passo": "removidas_custo_negativo_estorno", "linhas": int((~fora_janela & estorno).sum())},
    {"passo": "os_analiticas", "linhas": int(len(wo))},
    {"passo": "carretas_distintas", "linhas": int(wo["id_carreta"].nunique())},
    {"passo": "os_com_tipo_manutencao_MAINT", "linhas": int((wo["tipo_manutencao"] == "MAINT").sum())},
    {"passo": "os_sem_contrato_identificado", "linhas": int(wo["tipo_manutencao"].isna().sum())},
    {"passo": "descartada_franquia_km_mensal_contrato_D3_99.8pct_zeros", "linhas": 0},
])
curadoria.to_csv(TABLES / "02_curadoria_limpeza.csv", index=False)
print(curadoria.to_string(index=False))


                                                  passo  linhas
                                       linhas_originais  217217
                        removidas_fora_janela_2020_2025       5
                       removidas_custo_negativo_estorno     172
                                          os_analiticas  217040
                                     carretas_distintas    9585
                           os_com_tipo_manutencao_MAINT  194754
                           os_sem_contrato_identificado   16246
descartada_franquia_km_mensal_contrato_D3_99.8pct_zeros       0


## 2. Atributos estaticos do ativo (por carreta)

Atributos que descrevem a carreta (nao variam no tempo) sao consolidados por carreta:
categoricos pelo **primeiro valor nao nulo**; numericos pela **mediana**;
`data_entrada_servico` pelo **minimo**. Esses atributos ja vem incorporados ao CSV
consolidado (etapa anterior de *feature engineering*).

In [3]:
def mode_first(s):
    s = s.dropna()
    m = s.mode()
    return m.iloc[0] if len(m) else np.nan

attrs = wo.groupby("id_carreta").agg(
    descricao_carreta=("descricao_carreta", "first"),
    cod_montadora=("cod_montadora", "first"),
    flag_refrigerado=("flag_refrigerado", "first"),
    tailgate_flag=("tailgate_flag", "first"),
    unit_subtype=("unit_subtype", "first"),
    tire_size=("tire_size", "first"),
    suspension_type=("suspension_type", "first"),
    new_used_indicator=("new_used_indicator", "first"),
    ano_modelo=("ano_modelo", "median"),
    eixos=("eixos", "median"),
    comprimento=("comprimento", "median"),
    data_entrada_servico=("data_entrada_servico", "min"),
).reset_index()
print("atributos por carreta:", attrs.shape)
print("tailgate_flag valores unicos:", attrs["tailgate_flag"].dropna().unique(),
      "->", "CONSTANTE (variancia nula)" if attrs["tailgate_flag"].nunique() <= 1 else "ok")


atributos por carreta: (9585, 13)
tailgate_flag valores unicos: <StringArray>
['N']
Length: 1, dtype: str -> CONSTANTE (variancia nula)


## 3. Agregacao no grao carreta x ano

Cada carreta-ano recebe: custo interno **nominal** total, numero de OS, diversidade
de sistemas VMRS, participacao de OS preventivas (PM), odometro de fim de ano, VMRS
predominante e a regiao/provincia predominante da operacao.

In [4]:
ann = wo.groupby(["id_carreta", "ano"]).agg(
    custo_ano_nominal=("total_custo_interno", "sum"),
    n_os_ano=("id_os", "size"),
    n_sistemas_vmrs_distintos_ano=("vmrs", "nunique"),
    share_pm_ano=("vmrs", lambda s: float((s == "PM").mean())),
    km_acumulado_fim_ano=("km_acumulado_data_os", "max"),
    vmrs_predominante_ano=("vmrs", mode_first),
    regiao_operacao=("cod_local_os", mode_first),
    provincia_estado=("provincia_estado", mode_first),
    # --- contrato (2026-08-16) ---
    tipo_manutencao_ano=("tipo_manutencao", mode_first),
    share_maint_ano=("tipo_manutencao", lambda s: float((s == "MAINT").mean())),
    n_tipos_manutencao_ano=("tipo_manutencao", "nunique"),
    tempo_contrato_meses_fim_ano=("tempo_contrato_meses_ate_reparo", "max"),
    n_clientes_ano=("cod_cliente", "nunique"),
    cod_cliente_predominante_ano=("cod_cliente", mode_first),
).reset_index()

# ausencia de contrato e informacao, nao imputacao: a OS nao caiu no intervalo de nenhum contrato
ann["tipo_manutencao_ano"] = ann["tipo_manutencao_ano"].fillna("SEM_CONTRATO")
ann["cod_cliente_predominante_ano"] = ann["cod_cliente_predominante_ano"].fillna("SEM_CLIENTE")

# custo nominal por carreta x mes (insumo para a deflacao no notebook 04)
custo_mes = (wo.groupby(["id_carreta", "ano", "mes"])["total_custo_interno"].sum()
              .reset_index().rename(columns={"total_custo_interno": "custo_nominal_mes"}))
custo_mes.to_csv(DATA_PROCESSED / "custo_carreta_mes.csv", index=False)
print("carreta-ano observados (>=1 OS):", len(ann), "| carretas:", ann['id_carreta'].nunique())


carreta-ano observados (>=1 OS): 46229 | carretas: 9585


## 4. Grade carreta x ano (span ativo) e preenchimento

O grao analitico e **carreta x ano**. Para cada carreta considera-se o intervalo do
**primeiro ao ultimo ano** em que ela aparece na base (span ativo). Anos ativos sem
nenhuma OS recebem custo e contagens **zero** — representam um custo anual de
manutencao legitimamente nulo, e nao ausencia de dado. Regiao/provincia sao
propagadas (ffill/bfill) dentro da carreta; onde nunca ha informacao, `DESCONHECIDO`.


In [5]:
span = ann.groupby("id_carreta")["ano"].agg(ano_ini="min", ano_fim="max").reset_index()
span["anos"] = span.apply(lambda r: list(range(int(r.ano_ini), int(r.ano_fim) + 1)), axis=1)
grid = span[["id_carreta", "anos"]].explode("anos").rename(columns={"anos": "ano"})
grid["ano"] = grid["ano"].astype(int)

base = grid.merge(ann, on=["id_carreta", "ano"], how="left")
zero_cols = ["custo_ano_nominal", "n_os_ano", "n_sistemas_vmrs_distintos_ano", "share_pm_ano",
             "share_maint_ano", "n_tipos_manutencao_ano", "n_clientes_ano"]
base[zero_cols] = base[zero_cols].fillna(0)
base["vmrs_predominante_ano"] = base["vmrs_predominante_ano"].fillna("SEM_OS")
# ano ativo sem nenhuma OS: nao ha contrato observavel naquele ano (distinto de SEM_CONTRATO)
base["tipo_manutencao_ano"] = base["tipo_manutencao_ano"].fillna("SEM_OS")
base["cod_cliente_predominante_ano"] = base["cod_cliente_predominante_ano"].fillna("SEM_OS")

base = base.sort_values(["id_carreta", "ano"]).reset_index(drop=True)
for c in ["regiao_operacao", "provincia_estado"]:
    base[c] = base.groupby("id_carreta")[c].ffill()
    base[c] = base.groupby("id_carreta")[c].bfill()
    base[c] = base[c].fillna("DESCONHECIDO")

print("linhas grade (carreta x ano ativo):", len(base))
print("anos-ativos sem OS (custo anual = 0):", int((base['n_os_ano'] == 0).sum()),
      f"({(base['n_os_ano'] == 0).mean()*100:.1f}%)")


linhas grade (carreta x ano ativo): 47715
anos-ativos sem OS (custo anual = 0): 1486 (3.1%)


## 5. Atributos do ativo, idade e exposicao (odometro)

`idade_carreta` = ano de referencia menos ano de entrada em servico (fallback: ano do
modelo). `km_rodado_ano` = variacao do odometro de fim de ano em relacao ao ano ativo
anterior, com tratamento de resets/ruido (deltas negativos ou acima de 250.000 km ->
ausente).

In [6]:
base = base.merge(attrs, on="id_carreta", how="left")

base["idade_carreta"] = base["ano"] - base["data_entrada_servico"].dt.year
base.loc[base["idade_carreta"] < 0, "idade_carreta"] = np.nan
fb = base["idade_carreta"].isna()
base.loc[fb, "idade_carreta"] = (base["ano"] - base["ano_modelo"]).clip(lower=0)

# odometro de fim de ano: propaga ultimo conhecido para anos sem OS
base["km_acumulado_fim_ano"] = base.groupby("id_carreta")["km_acumulado_fim_ano"].ffill()
base.loc[base["km_acumulado_fim_ano"] < 0, "km_acumulado_fim_ano"] = np.nan
base["km_prev"] = base.groupby("id_carreta")["km_acumulado_fim_ano"].shift(1)
base["km_rodado_ano"] = base["km_acumulado_fim_ano"] - base["km_prev"]
base.loc[(base["km_rodado_ano"] < 0) | (base["km_rodado_ano"] > 250000), "km_rodado_ano"] = np.nan

diag_km = pd.DataFrame([
    {"metrica": "km_rodado_ano_ausente", "valor": int(base['km_rodado_ano'].isna().sum())},
    {"metrica": "km_rodado_ano_mediana", "valor": float(base['km_rodado_ano'].median())},
    {"metrica": "km_acumulado_fim_ano_mediana", "valor": float(base['km_acumulado_fim_ano'].median())},
])
diag_km.to_csv(TABLES / "02_diagnostico_km.csv", index=False)
base = base.drop(columns=["km_prev"])
print(diag_km.to_string(index=False))


                     metrica    valor
       km_rodado_ano_ausente   9893.0
       km_rodado_ano_mediana  13016.5
km_acumulado_fim_ano_mediana 110322.0


## 6. Historico defasado (anti-vazamento temporal)

Variaveis de historico usam **apenas** informacao de anos **anteriores** ao ano de
referencia, para permitir uso preditivo sem vazamento. As variaveis monetarias
defasadas (custo do ano anterior, custo acumulado) sao calculadas no notebook `04`,
apos a deflacao. Aqui derivam-se as contagens.

In [7]:
base = base.sort_values(["id_carreta", "ano"]).reset_index(drop=True)
g = base.groupby("id_carreta")
base["n_os_ano_anterior"] = g["n_os_ano"].shift(1)
base["n_os_acum_ate_ano_anterior"] = g["n_os_ano"].cumsum() - base["n_os_ano"]
base["anos_ativo_ate_ano_anterior"] = g.cumcount()
print(base[["id_carreta", "ano", "n_os_ano", "n_os_ano_anterior",
            "n_os_acum_ate_ano_anterior", "anos_ativo_ate_ano_anterior"]].head(8).to_string(index=False))

# --- contrato defasado e flags (2026-08-16) ---
# tempo de contrato conhecido no INICIO do ano = valor de fim do ano anterior (anti-vazamento)
base["tempo_contrato_meses_inicio_ano"] = g["tempo_contrato_meses_fim_ano"].shift(1)

# indicio de contrato novo no ano: mais de um tipo no ano OU queda no tempo de contrato
_caiu = base["tempo_contrato_meses_fim_ano"] < base["tempo_contrato_meses_inicio_ano"]
base["trocou_contrato_ano"] = ((base["n_tipos_manutencao_ano"] > 1) | _caiu.fillna(False)).astype(int)

# D6 (2026-08-16): populacao de modelagem = contrato com manutencao inclusa.
# Implementado como FLAG, seguindo o precedente da fase mensal — a base guarda todas
# as carreta-anos e a modelagem filtra pela flag (mantem baseline e EDA completos).
base["populacao_maint_flag"] = (base["tipo_manutencao_ano"] == "MAINT").astype(int)
print("populacao MAINT:", int(base["populacao_maint_flag"].sum()), "de", len(base),
      f"({base['populacao_maint_flag'].mean()*100:.1f}%)")
print(base["tipo_manutencao_ano"].value_counts(dropna=False).to_string())


 id_carreta  ano  n_os_ano  n_os_ano_anterior  n_os_acum_ate_ano_anterior  anos_ativo_ate_ano_anterior
         46 2020       1.0                NaN                         0.0                            0
         46 2021       1.0                1.0                         1.0                            1
         46 2022       0.0                1.0                         2.0                            2
         46 2023       2.0                0.0                         2.0                            3
         48 2020       2.0                NaN                         0.0                            0
         48 2021       2.0                2.0                         2.0                            1
         48 2022       1.0                2.0                         4.0                            2
         48 2023       2.0                1.0                         5.0                            3
populacao MAINT: 41739 de 47715 (87.5%)
tipo_manutencao_ano
MAINT        

## 7. Validacao e gravacao da base anual

Grava `data/processed/base_anual_carreta.csv` (custos nominais). O notebook `04`
adiciona a variavel resposta em valor **real** (CPI Canada).

In [8]:
assert base.duplicated(["id_carreta", "ano"]).sum() == 0, "grao carreta x ano nao unico"

validacao = pd.DataFrame([
    {"checagem": "linhas_base_anual", "valor": len(base)},
    {"checagem": "carretas", "valor": int(base['id_carreta'].nunique())},
    {"checagem": "anos", "valor": f"{int(base['ano'].min())}-{int(base['ano'].max())}"},
    {"checagem": "carreta_ano_sem_os", "valor": int((base['n_os_ano'] == 0).sum())},
    {"checagem": "custo_nominal_total_mi", "valor": round(base['custo_ano_nominal'].sum()/1e6, 3)},
    {"checagem": "grao_unico_carreta_ano", "valor": "OK"},
    {"checagem": "carreta_ano_populacao_maint", "valor": int(base['populacao_maint_flag'].sum())},
    {"checagem": "carreta_ano_sem_contrato", "valor": int((base['tipo_manutencao_ano'] == 'SEM_CONTRATO').sum())},
    {"checagem": "custo_nominal_maint_mi", "valor": round(base.loc[base['populacao_maint_flag'] == 1, 'custo_ano_nominal'].sum()/1e6, 3)},
    {"checagem": "cobertura_tempo_contrato_pct", "valor": round(base['tempo_contrato_meses_fim_ano'].notna().mean()*100, 1)},
])
validacao.to_csv(TABLES / "02_validacao_base_anual.csv", index=False)

dicionario = pd.DataFrame([
    ("id_carreta", "id", "Identificador da carreta"),
    ("ano", "temporal", "Ano de referencia (grao)"),
    ("custo_ano_nominal", "monetaria (nominal)", "Custo interno total no ano (CAD nominal); base da variavel resposta"),
    ("n_os_ano", "operacional/componente de Y", "Numero de OS internas no ano"),
    ("n_sistemas_vmrs_distintos_ano", "operacional", "Qtde de sistemas VMRS distintos com OS no ano"),
    ("share_pm_ano", "operacional", "Fracao de OS preventivas (VMRS=PM) no ano"),
    ("vmrs_predominante_ano", "operacional (cat.)", "Sistema VMRS mais frequente no ano"),
    ("km_acumulado_fim_ano", "exposicao", "Odometro no fim do ano"),
    ("km_rodado_ano", "exposicao", "Km rodado no ano (variacao do odometro; resets tratados)"),
    ("regiao_operacao", "geografica (cat.)", "Local/regiao predominante da OS (cod_local_os)"),
    ("provincia_estado", "geografica (cat.)", "Provincia predominante (parcial ~54%)"),
    ("cod_montadora", "ativo (cat.)", "Fabricante"),
    ("descricao_carreta", "ativo (cat.)", "Descricao/tipo da unidade (proxy de modelo/classe)"),
    ("ano_modelo", "ativo (quant.)", "Ano do modelo"),
    ("eixos", "ativo (quant.)", "Numero de eixos"),
    ("comprimento", "ativo (quant.)", "Comprimento (pes)"),
    ("flag_refrigerado", "ativo (cat.)", "Carreta refrigerada (Y/N)"),
    ("tailgate_flag", "ativo (cat.) - REMOVIDA", "Plataforma elevatoria; CONSTANTE (variancia nula) -> descartada"),
    ("unit_subtype", "ativo (cat.)", "Subtipo da unidade"),
    ("tire_size", "ativo (cat.)", "Tamanho de pneu"),
    ("suspension_type", "ativo (cat.)", "Tipo de suspensao"),
    ("new_used_indicator", "ativo (cat.)", "Novo/usado na aquisicao"),
    ("idade_carreta", "ativo derivada (quant.)", "Idade em anos no ano de referencia"),
    ("n_os_ano_anterior", "historico defasado", "OS no ano anterior"),
    ("n_os_acum_ate_ano_anterior", "historico defasado", "OS acumuladas ate o ano anterior"),
    ("anos_ativo_ate_ano_anterior", "historico defasado", "Anos ativos anteriores"),
    ("tipo_manutencao_ano", "contrato (cat.)", "Regime contratual predominante no ano (MAINT/NET/MIX/SEM_CONTRATO/SEM_OS)"),
    ("share_maint_ano", "contrato (quant.)", "Fracao de OS do ano sob regime MAINT"),
    ("n_tipos_manutencao_ano", "contrato (quant.)", "Qtde de regimes contratuais distintos no ano"),
    ("tempo_contrato_meses_fim_ano", "contrato (quant.)", "Maior tempo de contrato ate o reparo observado no ano"),
    ("tempo_contrato_meses_inicio_ano", "contrato defasado", "Tempo de contrato ao inicio do ano (fim de t-1); unico admissivel no cenario preditivo"),
    ("trocou_contrato_ano", "contrato (bin.)", "1 se houve indicio de novo contrato no ano"),
    ("n_clientes_ano", "contrato (quant.)", "Qtde de clientes distintos no ano"),
    ("cod_cliente_predominante_ano", "contrato (cat.) - DESCRITIVO", "Cliente com mais OS no ano; NAO entra como feature (597 categorias, risco de memorizacao)"),
    ("populacao_maint_flag", "flag de populacao", "1 = carreta-ano sob regime MAINT (populacao de modelagem, D6)"),
    ("franquia_km_mensal_contrato", "contrato - REMOVIDA", "99,8% dos valores preenchidos sao zero (variancia quase nula) -> descartada (D3)"),
], columns=["variavel", "tipo/papel", "descricao"])
dicionario.to_csv(TABLES / "02_dicionario_base_anual.csv", index=False)

base.to_csv(DATA_PROCESSED / "base_anual_carreta.csv", index=False)
print(validacao.to_string(index=False))
print("\nOK base anual salva:", (DATA_PROCESSED / 'base_anual_carreta.csv').name, base.shape)


                    checagem     valor
           linhas_base_anual     47715
                    carretas      9585
                        anos 2020-2025
          carreta_ano_sem_os      1486
      custo_nominal_total_mi    74.617
      grao_unico_carreta_ano        OK
 carreta_ano_populacao_maint     41739
    carreta_ano_sem_contrato      2779
      custo_nominal_maint_mi    70.497
cobertura_tempo_contrato_pct      91.1

OK base anual salva: base_anual_carreta.csv (47715, 36)
